# Chapter 5: Business Use Cases and Responsible Deployment

### Typical Business Applications of This Volume's Toolkit

1. **Customer support triage** — zero-shot or fine-tuned classification (Chapter 2) to route tickets automatically.
2. **Voice of customer** — sentiment analysis and topic extraction across reviews and NPS comments (Chapters 1–2).
3. **Internal knowledge assistant** — RAG (Chapters 3–4) over policy documents, past reports, and product manuals.
4. **Contract / report summarization** — turning long documents into executive bullet points (Chapter 3).
5. **Lead / opportunity scoring** from unstructured sales notes (Chapters 1–2 for entity/keyword extraction feeding into Volume 05's models).
6. **Automated report narration** — turning Volume 02–04's numeric analysis into a written executive summary via an LLM chain (Chapter 3).

### ⚠️ Responsible Deployment: Three Things to Always Consider

- **Hallucination risk** — an LLM can generate fluent, confident-sounding text that is simply factually wrong, especially when asked something outside its retrieved context. RAG (Chapter 4) substantially reduces this by grounding answers in real documents, but doesn't eliminate it entirely — always cite sources back to the retrieved chunks, and treat any ungrounded, purely generative claim (especially numbers) with real skepticism before it reaches a business decision.
- **Data privacy** — sending private company documents or customer data to a third-party LLM API has real data-governance implications; understand your organization's policy on this before building a prototype that pipes sensitive data to an external service.
- **Cost** — LLM API calls are billed per token (roughly, per word of input and output); a chatbot serving many users or processing long documents at scale can accumulate meaningful cost, and it's worth estimating this before committing to an architecture, not after deployment.

### 🎯 Business Challenge — Volume 09

*Task (from the source material):* extract entities/sentiment from a batch of customer reviews, then build a minimal RAG system over a handful of short business documents.

In [ ]:
# Part 1: sentiment + entity extraction over sample reviews
from transformers import pipeline
import spacy

reviews = [
    "The delivery was two weeks late and support never responded.",
    "Amazing product, exceeded expectations, will buy again!",
    "Priced too high compared to competitors like Acme Corp."
]

sentiment = pipeline("sentiment-analysis")
nlp = spacy.load("en_core_web_sm")

for r in reviews:
    s = sentiment(r)[0]
    ents = [(e.text, e.label_) for e in nlp(r).ents]
    print(f"'{r}'\n  Sentiment: {s['label']} ({s['score']:.2f})  |  Entities: {ents}\n")

# Part 2: a minimal RAG system over 3 short business documents
import chromadb
client = chromadb.Client()
collection = client.get_or_create_collection("policies")
collection.add(
    ids=["1", "2", "3"],
    documents=[
        "Refunds are available within 30 days of purchase with a valid receipt.",
        "The Premium support plan guarantees a response within 4 business hours.",
        "Employees are entitled to 18 days of paid annual leave per calendar year."
    ],
    metadatas=[{"topic": "returns"}, {"topic": "support"}, {"topic": "hr"}]
)

for question in ["How many days do I have to return something?", "How much annual leave do employees get?"]:
    result = collection.query(query_texts=[question], n_results=1)
    print(f"Q: {question}\nA (retrieved): {result['documents'][0][0]}\n")

*Note on time savings, worth writing up as the business memo the challenge asks for:* a manual review of 100 customer comments for sentiment and key entities might take an analyst 1–2 hours; the pipeline above processes them in seconds, freeing that analyst's time for interpreting patterns and recommending action rather than manual reading and tagging — precisely the "AutoML/AI as a force multiplier, not a replacement for judgment" theme that has run through this entire course.

---

## Volume 09 — What You Should Be Able to Do Now

- Extract named entities and build exact-match patterns with spaCy
- Use a Hugging Face pipeline for sentiment analysis and zero-shot classification with your own business categories
- Explain what a text embedding is and why semantically similar texts get numerically similar vectors
- Build a LangChain chain that composes a prompt, an LLM call, and an output parser
- Explain the two phases of RAG (indexing vs. retrieval) and build a minimal ChromaDB-backed retrieval example
- List the three responsible-deployment concerns (hallucination, privacy, cost) any real business LLM application must address

**Next: `10_Databases_Apps_and_GPU.md`**